# Environmental disclosure on a model hub

Runs entirely against the cached 3,000-model snapshot. No live queries, no GPU.

Set `CACHE` to your unzipped `hf_gov_cache_export` directory.


In [ ]:
# CELL 1 (replaces the original). Locates the cache wherever it is and unzips
# the export if only the .zip is present. Works in Colab and locally.

import json, re, pathlib, zipfile
import pandas as pd, numpy as np
from collections import Counter

def find_cache(explicit=None):
    """Return a directory containing models.jsonl, unzipping an export if needed."""
    if explicit:
        p = pathlib.Path(explicit)
        if (p / "models.jsonl").exists():
            return p
        raise FileNotFoundError(f"{p} has no models.jsonl")

    search = [pathlib.Path("."), pathlib.Path("/content"),
              pathlib.Path("/content/drive/MyDrive"), pathlib.Path.home()]

    # already-unzipped cache anywhere a couple of levels down
    for root in search:
        if not root.exists():
            continue
        for hit in list(root.glob("models.jsonl")) + list(root.glob("*/models.jsonl")) \
                 + list(root.glob("*/*/models.jsonl")):
            return hit.parent

    # otherwise find the export zip and unpack it
    for root in search:
        if not root.exists():
            continue
        for z in list(root.glob("hf_gov_cache*.zip")) + list(root.glob("*/hf_gov_cache*.zip")):
            dest = pathlib.Path("/content/hf_gov_cache") if pathlib.Path("/content").exists() \
                   else pathlib.Path("./hf_gov_cache")
            dest.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(z) as zf:
                zf.extractall(dest)
            print(f"unzipped {z.name} -> {dest}")
            for hit in list(dest.glob("models.jsonl")) + list(dest.glob("*/models.jsonl")):
                return hit.parent

    raise FileNotFoundError(
        "Could not find models.jsonl or hf_gov_cache*.zip.\n"
        "In Colab: click the folder icon, upload hf_gov_cache_export.zip, re-run.\n"
        "Or set it yourself: CACHE = find_cache('/content/my_folder')")

CACHE = find_cache()          # or find_cache('/content/hf_gov_cache')
print("cache:", CACHE.resolve())

models  = [json.loads(l) for l in open(CACHE / "models.jsonl", encoding="utf-8")]
readmes = {json.loads(l)["id"]: json.loads(l)
           for l in open(CACHE / "readmes.jsonl", encoding="utf-8")}

df = pd.DataFrame(models)
df["downloads"] = df.downloads.fillna(0)
df["tags"] = df.tags.apply(lambda t: t or [])

print(f"loaded {len(df)} models, {sum(1 for r in readmes.values() if r.get('readme'))} cards with text")
assert len(df) > 0 and "tags" in df, "cache loaded but looks empty"

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    ph = k/n; d = 1 + z*z/n
    c = (ph + z*z/(2*n))/d
    h = z*np.sqrt(ph*(1-ph)/n + z*z/(4*n*n))/d
    return round((c-h)*100, 2), round((c+h)*100, 2)


In [ ]:
print("="*70); print("RQ1  Structured emissions disclosure"); print("="*70)
df["co2_tag"] = df.tags.apply(lambda t: any(x == "co2_eq_emissions" for x in t))
k, n = int(df.co2_tag.sum()), len(df)
print(f"  models declaring co2_eq_emissions: {k}/{n} = {k/n*100:.2f}%  CI95 {wilson(k,n)}")
print("  the disclosing models:")
for _, r in df[df.co2_tag].sort_values("downloads", ascending=False).iterrows():
    print(f"    {r['id'][:55]:55s} downloads={int(r.downloads):>10,}")

print("\n" + "="*70); print("RQ2  Efficiency-oriented artifacts (the thing being optimized)"); print("="*70)
EFF_TAGS = {"gguf":"GGUF","onnx":"ONNX","awq":"AWQ","gptq":"GPTQ","quantized":"quantized",
            "4-bit":"4-bit","8-bit":"8-bit","bitsandbytes":"bitsandbytes","mlx":"MLX",
            "openvino":"OpenVINO","tensorrt":"TensorRT","exl2":"EXL2","quantization":"quantization"}
low = df.tags.apply(lambda t: {x.lower() for x in t})
for key, label in EFF_TAGS.items():
    c = int(low.apply(lambda s: key in s).sum())
    if c: print(f"    {label:14s} {c:5d} models ({c/n*100:4.1f}%)")
df["efficiency_artifact"] = low.apply(lambda s: bool(s & set(EFF_TAGS)))
k2 = int(df.efficiency_artifact.sum())
print(f"\n  any efficiency/compression tag: {k2}/{n} = {k2/n*100:.1f}%  CI95 {wilson(k2,n)}")

declared = df[df.base_models.apply(len) > 0]
rel = Counter(declared.relation_kind)
print(f"\n  derivation type among the {len(declared)} models declaring a parent:")
for kk, vv in rel.most_common():
    print(f"    {kk:10s} {vv:5d} ({vv/len(declared)*100:4.1f}%)")
q = rel.get("quantized", 0)
print(f"  quantization share of declared derivations: {q}/{len(declared)} = {q/len(declared)*100:.1f}%"
      f"  CI95 {wilson(q,len(declared))}")

print("\n" + "="*70); print("RQ3  Energy and carbon reporting in card prose"); print("="*70)
FRONT = re.compile(r"^---\s*\r?\n.*?\r?\n---\s*\r?\n", re.DOTALL)
PATTERNS = {
 "carbon_or_co2":   r"\b(carbon footprint|carbon emission|co2|co\u2082|greenhouse)\b",
 "energy_units":    r"\b(kwh|kilowatt|joule|watt-hour|\bwh\b)\b",
 "power_draw":      r"\b(tdp|power draw|watts?)\b",
 "compute_hours":   r"\b(gpu[- ]hours?|tpu[- ]hours?|a100[- ]hours?|training time|compute hours?)\b",
 "hardware_named":  r"\b(a100|h100|v100|tpu v[0-9]|rtx ?[0-9]{4}|l40s?|mi[23]00)\b",
 "efficiency_claim":r"\b(more efficient|energy[- ]efficient|lower memory|less vram|runs on (a )?(cpu|laptop|phone)|smaller footprint)\b",
}
rows = []
for mid in df.id:
    rec = readmes.get(mid)
    if not rec or not rec.get("readme"): continue
    body = FRONT.sub("", rec["readme"], count=1)
    hit = {k2_: bool(re.search(p, body, re.I)) for k2_, p in PATTERNS.items()}
    hit["id"] = mid; hit["chars"] = len(body)
    rows.append(hit)
pr = pd.DataFrame(rows)
N = len(pr)
print(f"  cards analyzed: {N}")
for c in PATTERNS:
    k3 = int(pr[c].sum()); print(f"    {c:18s} {k3:4d} ({k3/N*100:5.1f}%)  CI95 {wilson(k3,N)}")
pr["any_env"] = pr[["carbon_or_co2","energy_units","power_draw","compute_hours"]].any(axis=1)
k4 = int(pr.any_env.sum())
print(f"\n  ANY quantitative energy/carbon/compute term: {k4}/{N} = {k4/N*100:.1f}%  CI95 {wilson(k4,N)}")
k5 = int(pr.efficiency_claim.sum())
print(f"  makes an efficiency CLAIM but no quantity: "
      f"{int((pr.efficiency_claim & ~pr.any_env).sum())}/{N} "
      f"= {(pr.efficiency_claim & ~pr.any_env).mean()*100:.1f}%")
print(f"  efficiency claim of any kind: {k5}/{N} = {k5/N*100:.1f}%")

print("\n" + "="*70); print("RQ4  Disclosure among efficiency artifacts specifically"); print("="*70)
eff = df[df.efficiency_artifact]
print(f"  efficiency artifacts: {len(eff)}")
print(f"    declaring co2_eq_emissions: {int(eff.co2_tag.sum())} ({eff.co2_tag.mean()*100:.2f}%)")
pr2 = pr.merge(df[["id","efficiency_artifact"]], on="id")
for flag, g in pr2.groupby("efficiency_artifact"):
    lbl = "efficiency artifact" if flag else "other model"
    print(f"    {lbl:20s} n={len(g):4d}  any energy term={g.any_env.mean()*100:5.1f}%  "
          f"efficiency claim={g.efficiency_claim.mean()*100:5.1f}%")

print("\n" + "="*70); print("RQ5  Exposure: downloads attached to undocumented artifacts"); print("="*70)
tot = df.downloads.sum()
eff_dl = eff.downloads.sum()
disc_dl = df[df.co2_tag].downloads.sum()
print(f"  total monthly downloads in sample : {int(tot):,}")
print(f"  held by efficiency artifacts      : {int(eff_dl):,} ({eff_dl/tot*100:.1f}%)")
print(f"  held by models disclosing CO2e    : {int(disc_dl):,} ({disc_dl/tot*100:.3f}%)")
print(f"  downloads with NO emissions figure: {int(tot-disc_dl):,} ({(tot-disc_dl)/tot*100:.2f}%)")

out = pathlib.Path("/home/claude/esr/results")
out.mkdir(exist_ok=True)
pr.to_csv(out/"prose_env_flags.csv", index=False)
df[["id","downloads","co2_tag","efficiency_artifact","relation_kind"]].to_csv(out/"model_env_flags.csv", index=False)
print("\n  wrote results/ tables")


# ---------------------------------------------------------------------------
# Comparative tests. Run after the blocks above.
# ---------------------------------------------------------------------------
from scipy import stats

def or_ci(table, z=1.96):
    """Odds ratio with Woolf log CI. table = [[a,b],[c,d]] as counts."""
    (a, b), (c, d) = table
    if 0 in (a, b, c, d):
        a, b, c, d = a+0.5, b+0.5, c+0.5, d+0.5   # Haldane correction
    orv = (a*d)/(b*c)
    se = np.sqrt(1/a + 1/b + 1/c + 1/d)
    return orv, np.exp(np.log(orv) - z*se), np.exp(np.log(orv) + z*se)

pr2 = pr.merge(df[["id", "efficiency_artifact"]], on="id")
print("\n" + "="*70); print("RQ6  Claim versus quantification asymmetry"); print("="*70)

# rows ordered [efficiency artifact, other] so the OR reads "artifact vs other"
for label, col in [("makes an efficiency claim", "efficiency_claim"),
                   ("quantifies energy/carbon", "any_env")]:
    eff_y = int(pr2[pr2.efficiency_artifact][col].sum())
    eff_n = int((~pr2[pr2.efficiency_artifact][col]).sum())
    oth_y = int(pr2[~pr2.efficiency_artifact][col].sum())
    oth_n = int((~pr2[~pr2.efficiency_artifact][col]).sum())
    orv, lo, hi = or_ci([[eff_y, eff_n], [oth_y, oth_n]])
    _, p = stats.fisher_exact([[eff_y, eff_n], [oth_y, oth_n]])
    print(f"  {label:26s} artifacts {eff_y}/{eff_y+eff_n} ({eff_y/(eff_y+eff_n)*100:4.1f}%)  "
          f"others {oth_y}/{oth_y+oth_n} ({oth_y/(oth_y+oth_n)*100:4.1f}%)")
    print(f"    OR={orv:.2f} [{lo:.2f}, {hi:.2f}]  Fisher p={p:.4f}"
          f"{'  (marginal)' if 0.01 < p < 0.06 else ''}")

print("\n  Reading: compression artifacts are more likely to ASSERT efficiency but no")
print("  more likely to QUANTIFY it. The claim rises, the measurement does not.")

print("\n" + "="*70); print("NOT AVAILABLE from this cache"); print("="*70)
print("  last_modified was not persisted by the Stage A normalizer (field name")
print("  differs in huggingface_hub 1.x), so no temporal analysis is possible")
print("  without a small re-fetch. Any claim that disclosure is a declining or")
print("  legacy practice is therefore NOT supported by these data.")
